In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

if xgb.__version__ != "3.2.0":
    raise RuntimeError(f"B05 parity requires XGBoost 3.2.0; found {xgb.__version__}")

_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents]
             if (p / "data/processed/split_trip_ids.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate SupplyShield project root")
DATA = ROOT / "data/processed"
EXP_DAY2 = ROOT / "experiments/day2/master_split"
EXP_DAY3 = ROOT / "experiments/day3/master_split_v2"
DAY5 = ROOT / "experiments/day5"
RISK_PATH = DAY5 / "risk_table_master_split_v1.csv"
MANIFEST_PATH = DAY5 / "risk_table_master_split_v1_manifest.json"
SUCCESS_PATH = DAY5 / "risk_table_master_split_v1_SUCCESS.json"


In [ ]:
# Load only the authoritative master-split/core inputs and current B02/B03 artifacts.
split_ids = pd.read_csv(DATA / "split_trip_ids.csv", parse_dates=["trip_creation_time"])
legs = pd.read_csv(DATA / "legs.csv", parse_dates=["od_start_time", "trip_creation_time"])
legs["od_end_time"] = pd.to_datetime(legs["od_end_time"], format="mixed")
core_train = pd.read_csv(DATA / "core_train_legs.csv", parse_dates=["od_start_time", "trip_creation_time"])
core_train["od_end_time"] = pd.to_datetime(core_train["od_end_time"], format="mixed")
core_edges = pd.read_csv(DATA / "core_edges.csv")

MODEL_PATH = ROOT / "models/residual_xgb_plain_master_split_v1.json"
CARD_PATH = EXP_DAY2 / "final_model_card_master_split_v1.json"
B02_PRED_PATH = DATA / "preds_cal_test_master_split_v1.csv"
PARAMS_PATH = EXP_DAY3 / "final/conformal_params_method_c_master_split_v2.json"
B03_MANIFEST_PATH = EXP_DAY3 / "run_manifest.json"
B03_SUCCESS_PATH = EXP_DAY3 / "_SUCCESS.json"
B03_TEST_PATH = EXP_DAY3 / "final/test_intervals_method_c_master_split_v2.csv"
B03_METRICS_PATH = EXP_DAY3 / "evaluation/interval_comparison_all_methods.csv"

for required in [MODEL_PATH, CARD_PATH, B02_PRED_PATH, PARAMS_PATH, B03_MANIFEST_PATH,
                 B03_SUCCESS_PATH, B03_TEST_PATH, B03_METRICS_PATH]:
    if not required.is_file():
        raise FileNotFoundError(required)

model_card = json.loads(CARD_PATH.read_text(encoding="utf-8"))
conformal_params = json.loads(PARAMS_PATH.read_text(encoding="utf-8"))
b03_manifest = json.loads(B03_MANIFEST_PATH.read_text(encoding="utf-8"))
b03_success = json.loads(B03_SUCCESS_PATH.read_text(encoding="utf-8"))
b02_saved = pd.read_csv(B02_PRED_PATH, parse_dates=["od_start_time"])
b03_test_saved = pd.read_csv(B03_TEST_PATH, parse_dates=["od_start_time"])
b03_metrics = pd.read_csv(B03_METRICS_PATH)

assert b03_manifest["status"] == b03_success["status"] == "complete"
assert conformal_params["experiment"] == "day3_master_split_v2"
assert conformal_params["input_prediction_file"].replace("\\", "/") == "data/processed/preds_cal_test_master_split_v1.csv"
assert b03_manifest["input"].replace("\\", "/") == "data/processed/preds_cal_test_master_split_v1.csv"
assert model_card["split_source"].replace("\\", "/") == "data/processed/split_trip_ids.csv"
assert not any((DAY5 / n).exists() for n in [RISK_PATH.name, MANIFEST_PATH.name, SUCCESS_PATH.name]), \
    "Refusing to overwrite an existing B05 output; choose a new versioned name"


In [ ]:
# Reconstruct B02's exact master-split as-of feature pipeline and verify the saved predictions.
FEATURES = model_card["features"]
EXPECTED_FEATURES = [
    "osrm_time_min", "osrm_distance_km", "osrm_speed_kmh", "log_osrm_time",
    "start_hour", "start_dow", "is_weekend", "is_ftl", "lane_base", "lane_history_count"
]
assert FEATURES == EXPECTED_FEATURES

split_map = split_ids.set_index("trip_uuid")["split"]
assert split_ids["trip_uuid"].is_unique
assert set(split_ids["split"].unique()) == {"train", "calibration", "test"}
legs["split"] = legs["trip_uuid"].map(split_map)
assert legs["split"].notna().all()
train = legs[legs["split"] == "train"].copy()
cal = legs[legs["split"] == "calibration"].copy()
test = legs[legs["split"] == "test"].copy()

train_trip_dates = train.groupby("trip_uuid")["trip_creation_time"].min().sort_values()
fit_trip_ids = train_trip_dates.index[:int(len(train_trip_dates) * 0.8)]
fit_df = train[train["trip_uuid"].isin(fit_trip_ids)].copy()
g_fit = float(fit_df["delay_ratio"].median())

def add_historical_lane_features_asof(target_df, history_df):
    result = target_df.copy()
    lane_cols = ["source_center", "destination_center"]
    result["lane_median_ratio"] = 1.0
    result["lane_history_count"] = 0
    history_groups = {
        key: group.sort_values("od_end_time")
        for key, group in history_df.groupby(lane_cols, sort=False)
    }
    for key, target_group in result.groupby(lane_cols, sort=False):
        if key not in history_groups:
            continue
        hist = history_groups[key]
        end_times = hist["od_end_time"].to_numpy(dtype="datetime64[ns]")
        delays = hist["delay_ratio"].to_numpy()
        for idx, row in target_group.iterrows():
            start_time = np.datetime64(row["od_start_time"], "ns")
            count = np.searchsorted(end_times, start_time, side="right")
            result.loc[idx, "lane_history_count"] = count
            if count > 0:
                result.loc[idx, "lane_median_ratio"] = np.median(delays[:count])
    return result

def build_features(frame):
    start = pd.to_datetime(frame["od_start_time"])
    X = pd.DataFrame(index=frame.index)
    X["osrm_time_min"] = frame["osrm_time_min"]
    X["osrm_distance_km"] = frame["osrm_distance_km"]
    X["osrm_speed_kmh"] = frame["osrm_distance_km"] / (frame["osrm_time_min"] / 60)
    X["log_osrm_time"] = np.log1p(frame["osrm_time_min"])
    X["start_hour"] = start.dt.hour
    X["start_dow"] = start.dt.dayofweek
    X["is_weekend"] = (start.dt.dayofweek >= 5).astype(int)
    X["is_ftl"] = (frame["route_type"] == "FTL").astype(int)
    return X

def base_pred(frame):
    return frame["lane_median_ratio"].where(frame["lane_history_count"] > 0, g_fit).to_numpy()

def make_X(frame):
    X = build_features(frame)[FEATURES[:8]].copy()
    X["lane_base"] = base_pred(frame)
    X["lane_history_count"] = frame["lane_history_count"].to_numpy()
    return X[FEATURES]

cal_v2 = add_historical_lane_features_asof(cal, train)
test_history = pd.concat([train, cal], ignore_index=True)
test_v2 = add_historical_lane_features_asof(test, test_history)
model = xgb.XGBRegressor()
model.load_model(str(MODEL_PATH))
assert model.best_iteration == int(model_card["best_iteration"]) == 134
assert model.get_booster().feature_names == FEATURES
assert model.get_booster().num_boosted_rounds() == 235

reproduced_parts = []
for pile, frame in [("cal", cal_v2), ("test", test_v2)]:
    X = make_X(frame)
    assert list(X.columns) == FEATURES
    residual = model.predict(X)  # B02 cell 83: sklearn wrapper default uses saved best_iteration.
    base = base_pred(frame)
    reproduced_parts.append(pd.DataFrame({
        "trip_uuid": frame["trip_uuid"].to_numpy(),
        "source_center": frame["source_center"].to_numpy(),
        "destination_center": frame["destination_center"].to_numpy(),
        "pile": pile,
        "base_pred_reproduced": base,
        "residual_reproduced": residual,
        "pred_ratio_reproduced": base + residual,
        "lane_history_count_reproduced": frame["lane_history_count"].to_numpy(),
    }))

b02_reproduced = pd.concat(reproduced_parts, ignore_index=True)
PRED_KEYS = ["trip_uuid", "source_center", "destination_center", "pile"]
b02_join = b02_reproduced.merge(b02_saved, on=PRED_KEYS, how="outer", validate="one_to_one", indicator=True)
assert len(b02_join) == 8695 and (b02_join["_merge"] == "both").all()
assert np.array_equal(b02_join["lane_history_count_reproduced"], b02_join["lane_history_count"])
BASE_MAX_DIFF = float(np.max(np.abs(b02_join["base_pred_reproduced"] - b02_join["base_pred"])))
RESIDUAL_MAX_DIFF = float(np.max(np.abs(b02_join["residual_reproduced"] - (b02_join["pred_ratio"] - b02_join["base_pred"]))))
B02_MAX_DIFF = float(np.max(np.abs(b02_join["pred_ratio_reproduced"] - b02_join["pred_ratio"])))
assert BASE_MAX_DIFF <= 1e-6 and RESIDUAL_MAX_DIFF <= 1e-6 and B02_MAX_DIFF <= 1e-6

B02_METRICS = {}
for pile, label in [("cal", "calibration"), ("test", "test")]:
    part = b02_saved[b02_saved["pile"] == pile]
    repro = b02_join[b02_join["pile"] == pile]
    mae = float(mean_absolute_error(part["y_ratio"], repro["pred_ratio_reproduced"]))
    saved_mae = float(mean_absolute_error(part["y_ratio"], part["pred_ratio"]))
    card_mae = float(model_card["results"][f"{label} - residual XGBoost"]["MAE_ratio"])
    assert abs(mae - saved_mae) <= 1e-12 and abs(mae - card_mae) <= 0.00005
    B02_METRICS[pile] = {"rows": int(len(part)), "mae_ratio": mae, "saved_mae_ratio": saved_mae, "card_mae_ratio": card_mae}


In [ ]:
# Rebuild Method C intervals from saved B03 parameters and verify every saved test interval.
cal_pred = b02_saved[b02_saved["pile"] == "cal"].copy()
test_pred = b02_saved[b02_saved["pile"] == "test"].copy()
DISTANCE_EDGES = [float(x) for x in conformal_params["distance_edges_km"]]
assert np.allclose(DISTANCE_EDGES, [31.21256666666666, 60.86566666666664], rtol=0, atol=1e-12)
cal_quantiles = cal_pred["osrm_distance_km"].quantile([1/3, 2/3]).to_numpy()
assert np.allclose(cal_quantiles, DISTANCE_EDGES, rtol=0, atol=1e-12)

def add_method_c_group(frame):
    out = frame.copy()
    out["dist_bucket"] = pd.cut(
        out["osrm_distance_km"], [-np.inf, DISTANCE_EDGES[0], DISTANCE_EDGES[1], np.inf],
        labels=["short", "medium", "long"], include_lowest=True
    ).astype("object")
    out["grp"] = out["route_type"].astype(str) + "|" + out["dist_bucket"].astype(str)
    return out

def apply_method_c(frame):
    out = frame.copy()
    for level in ["50%", "80%", "90%"]:
        info = conformal_params["levels"][level]
        offsets = [info["group_offsets"].get(g, info["global_offsets"]) for g in out["grp"]]
        lower_offset = np.array([x[0] for x in offsets], dtype=float)
        upper_offset = np.array([x[1] for x in offsets], dtype=float)
        tag = level.replace("%", "")
        out[f"lo{tag}"] = np.clip(out["pred_ratio"].to_numpy() + lower_offset, 0, None)
        out[f"hi{tag}"] = out["pred_ratio"].to_numpy() + upper_offset
    return out

test_method_c = apply_method_c(add_method_c_group(test_pred))
INTERVAL_KEYS = ["trip_uuid", "source_center", "destination_center", "od_start_time"]
b03_join = test_method_c.merge(b03_test_saved, on=INTERVAL_KEYS, how="outer", validate="one_to_one", indicator=True, suffixes=("_reproduced", "_saved"))
assert len(b03_join) == 4394 and (b03_join["_merge"] == "both").all()
assert (b03_join["grp_reproduced"] == b03_join["grp_saved"]).all()
assert (b03_join["dist_bucket_reproduced"] == b03_join["dist_bucket_saved"]).all()
assert np.allclose(b03_join["pred_ratio_reproduced"], b03_join["pred_ratio_saved"], rtol=0, atol=1e-12)
for tag in ["50", "80", "90"]:
    assert np.allclose(b03_join[f"lo{tag}"], b03_join[f"lower_{tag}"], rtol=0, atol=1e-12)
    assert np.allclose(b03_join[f"hi{tag}"], b03_join[f"upper_{tag}"], rtol=0, atol=1e-12)

B03_COVERAGE = {}
for tag, level in [("50", "50%"), ("80", "80%"), ("90", "90%")]:
    y = b03_join["y_ratio_reproduced"] if "y_ratio_reproduced" in b03_join else b03_join["y_ratio"]
    coverage = float(((y >= b03_join[f"lo{tag}"]) & (y <= b03_join[f"hi{tag}"])).mean())
    metric_row = b03_metrics[(b03_metrics["method"] == "C grouped (route x distance)") & (b03_metrics["level"] == level)]
    assert len(metric_row) == 1 and abs(coverage - float(metric_row.iloc[0]["coverage"])) <= 1e-12
    B03_COVERAGE[level] = {"test_rows": int(len(b03_join)), "test_trips": int(b03_join["trip_uuid"].nunique()), "coverage": coverage, "target": float(metric_row.iloc[0]["target"])}


In [ ]:
# Training-only representative lane-route rows at the common as-of cutoff.
CUTOFF = pd.Timestamp("2018-09-29 00:00:00")
LANE_KEYS = ["source_center", "destination_center"]
ROUTE_KEYS = LANE_KEYS + ["route_type"]
EDGE_KEYS = set(map(tuple, core_edges[LANE_KEYS].to_numpy()))
assert len(core_edges) == 292 and core_edges.duplicated(LANE_KEYS).sum() == 0

core_split = core_train["trip_uuid"].map(split_map)
assert core_split.notna().all() and set(core_split.unique()) == {"train"}
assert not core_train["delay_ratio"].isna().any()
assert not core_train["od_end_time"].isna().any()
core_pairs = list(map(tuple, core_train[LANE_KEYS].to_numpy()))
assert set(core_pairs) <= EDGE_KEYS

# Prove core_train_legs is exactly the master-training leg population on frozen directed lanes.
train_pairs = list(map(tuple, train[LANE_KEYS].to_numpy()))
train_core = train.loc[[pair in EDGE_KEYS for pair in train_pairs]].copy()
MATCH_COLS = ["trip_uuid", "source_center", "destination_center", "od_start_time", "od_end_time"]
assert len(train_core) == len(core_train)
left = train_core[MATCH_COLS].astype(str).sort_values(MATCH_COLS).reset_index(drop=True)
right = core_train[MATCH_COLS].astype(str).sort_values(MATCH_COLS).reset_index(drop=True)
assert left.equals(right), "core_train_legs differs from the master-training rows on frozen lanes"

eligible = core_train[core_train["od_end_time"] <= CUTOFF].copy()
assert len(eligible) == len(core_train), "Some core training history ends after the common cutoff"
assert set(eligible["trip_uuid"].map(split_map).unique()) == {"train"}
eligible["_start"] = pd.to_datetime(eligible["od_start_time"])
eligible["start_hour"] = eligible["_start"].dt.hour.astype(int)
eligible["start_dow"] = eligible["_start"].dt.dayofweek.astype(int)

# B02 history is lane-only, so lane_base/count aggregate all route types on a directed lane.
lane_history = (eligible.groupby(LANE_KEYS, sort=False)
                .agg(lane_median_ratio=("delay_ratio", "median"),
                     lane_history_count=("delay_ratio", "size"))
                .reset_index())

# Representative OSRM values and departure context are route-specific.
route_reps = (eligible.groupby(ROUTE_KEYS, sort=False)
              .agg(osrm_time_min=("osrm_time_min", "median"),
                   osrm_distance_km=("osrm_distance_km", "median"),
                   representative_training_legs=("trip_uuid", "size"))
              .reset_index())
context_counts = (eligible.groupby(ROUTE_KEYS + ["start_hour", "start_dow"], sort=False)
                  .size().rename("context_count").reset_index())
context_counts = context_counts.sort_values(
    ROUTE_KEYS + ["context_count", "start_hour", "start_dow"],
    ascending=[True, True, True, False, True, True], kind="mergesort"
)
modal_context = context_counts.drop_duplicates(ROUTE_KEYS, keep="first")[
    ROUTE_KEYS + ["start_hour", "start_dow", "context_count"]
]
representatives = route_reps.merge(modal_context, on=ROUTE_KEYS, how="left", validate="one_to_one")
representatives = representatives.merge(lane_history, on=LANE_KEYS, how="left", validate="many_to_one")
assert len(representatives) == 296 and representatives.duplicated(ROUTE_KEYS).sum() == 0
assert representatives["lane_history_count"].notna().all() and (representatives["lane_history_count"] > 0).all()
assert representatives["route_type"].notna().all()
representatives["is_weekend"] = (representatives["start_dow"] >= 5).astype(int)
representatives["is_ftl"] = (representatives["route_type"] == "FTL").astype(int)
representatives["osrm_speed_kmh"] = representatives["osrm_distance_km"] / (representatives["osrm_time_min"] / 60)
representatives["log_osrm_time"] = np.log1p(representatives["osrm_time_min"])
representatives["lane_base"] = representatives["lane_median_ratio"].where(
    representatives["lane_history_count"] > 0, g_fit
)
representatives["prediction_timestamp"] = CUTOFF.strftime("%Y-%m-%d %H:%M:%S")

assert set(map(tuple, representatives[LANE_KEYS].drop_duplicates().to_numpy())) == EDGE_KEYS
assert representatives["representative_training_legs"].sum() == len(eligible)
assert representatives["lane_history_count"].sum() >= len(eligible)


In [ ]:
# Score each representative leg with the current B02 sklearn model and apply Method C offsets.
X_lane = representatives[FEATURES].copy()
assert list(X_lane.columns) == model_card["features"] == model.get_booster().feature_names
assert np.isfinite(X_lane.to_numpy(dtype=float)).all()
residual_lane = model.predict(X_lane)  # default sklearn prediction uses best_iteration=134
lane_pred = representatives["lane_base"].to_numpy() + residual_lane

lane_scored = add_method_c_group(representatives)
lane_scored["residual_prediction"] = residual_lane
lane_scored["pred_ratio"] = lane_pred
lane_scored = apply_method_c(lane_scored)
lane_scored["osrm_time_h"] = lane_scored["osrm_time_min"] / 60
lane_scored["pred_time_h"] = lane_scored["pred_ratio"] * lane_scored["osrm_time_h"]
lane_scored["pred_excess_h"] = (lane_scored["pred_ratio"] - 1) * lane_scored["osrm_time_h"]
for tag in ["50", "80", "90"]:
    lane_scored[f"excess_lo{tag}_h"] = (lane_scored[f"lo{tag}"] - 1) * lane_scored["osrm_time_h"]
    lane_scored[f"excess_hi{tag}_h"] = (lane_scored[f"hi{tag}"] - 1) * lane_scored["osrm_time_h"]

OUTPUT_COLUMNS = [
    "source_center", "destination_center", "route_type", "prediction_timestamp",
    "dist_bucket", "grp", "representative_training_legs", "context_count",
    "lane_history_count", "lane_base", "start_hour", "start_dow", "is_weekend", "is_ftl",
    "osrm_time_min", "osrm_time_h", "osrm_distance_km", "osrm_speed_kmh", "log_osrm_time",
    "residual_prediction", "pred_ratio",
    "lo50", "hi50", "lo80", "hi80", "lo90", "hi90",
    "pred_time_h", "pred_excess_h",
    "excess_lo50_h", "excess_hi50_h", "excess_lo80_h", "excess_hi80_h", "excess_lo90_h", "excess_hi90_h"
]
risk_table = lane_scored[OUTPUT_COLUMNS].copy()


In [ ]:
# Validate representative lane-route scores independently of held-out B02/B03 evaluation.
assert len(risk_table) == 296
assert risk_table.duplicated(ROUTE_KEYS).sum() == 0
assert risk_table[ROUTE_KEYS].notna().all().all()
assert set(map(tuple, risk_table[LANE_KEYS].drop_duplicates().to_numpy())) == EDGE_KEYS
assert risk_table["route_type"].nunique() >= 1
assert risk_table["prediction_timestamp"].eq("2018-09-29 00:00:00").all()
assert (eligible["od_end_time"] <= CUTOFF).all()
assert set(eligible["trip_uuid"].map(split_map).unique()) == {"train"}
assert np.isfinite(risk_table.select_dtypes(include=[np.number]).to_numpy(dtype=float)).all()
assert not risk_table.isna().any().any()
assert not any(c in risk_table.columns for c in ["trip_uuid", "y_ratio", "delay_ratio", "actual_time_min"])
assert list(X_lane.columns) == FEATURES
assert risk_table["lane_history_count"].min() >= 3
assert risk_table["dist_bucket"].isin(["short", "medium", "long"]).all()
assert (risk_table["lo50"] >= 0).all() and (risk_table["lo80"] >= 0).all() and (risk_table["lo90"] >= 0).all()
assert (risk_table["lo50"] <= risk_table["pred_ratio"]).all() and (risk_table["pred_ratio"] <= risk_table["hi50"]).all()
assert (risk_table["lo80"] <= risk_table["pred_ratio"]).all() and (risk_table["pred_ratio"] <= risk_table["hi80"]).all()
assert (risk_table["lo90"] <= risk_table["pred_ratio"]).all() and (risk_table["pred_ratio"] <= risk_table["hi90"]).all()
assert (risk_table["lo50"] >= risk_table["lo80"] - 1e-12).all()
assert (risk_table["lo80"] >= risk_table["lo90"] - 1e-12).all()
assert (risk_table["hi50"] <= risk_table["hi80"] + 1e-12).all()
assert (risk_table["hi80"] <= risk_table["hi90"] + 1e-12).all()

# In-memory CSV round-trip check; no output path is touched until this passes.
_roundtrip = pd.read_csv(StringIO(risk_table.to_csv(index=False)))
assert _roundtrip.shape == risk_table.shape
assert _roundtrip[ROUTE_KEYS].equals(risk_table[ROUTE_KEYS].reset_index(drop=True))
assert np.allclose(_roundtrip["pred_ratio"], risk_table["pred_ratio"], rtol=1e-12, atol=1e-12)
assert np.allclose(_roundtrip[["lo50", "hi50", "lo80", "hi80", "lo90", "hi90"]],
                   risk_table[["lo50", "hi50", "lo80", "hi80", "lo90", "hi90"]], rtol=1e-12, atol=1e-12)

print("PASS representative rows:", len(risk_table), "| directed lanes:", len(EDGE_KEYS),
      "| route variants:", len(risk_table) - len(EDGE_KEYS))
print("PASS B02 prediction parity: rows=8695 | max base diff=", BASE_MAX_DIFF,
      "| max residual diff=", RESIDUAL_MAX_DIFF, "| max ratio diff=", B02_MAX_DIFF)
print("PASS B03 Method C test interval parity: rows=4394 | coverage:", B03_COVERAGE)


In [ ]:
# Keep test evaluation separate from the representative lane-route score table.
print("B02 held-out evaluation (leg rows):", B02_METRICS)
print("B03 Method C held-out test coverage (leg rows, not lane coverage):", B03_COVERAGE)
print("Representative lane-route rows are model scores for modal training departure contexts.")
print("Their conformal bounds are representative-leg intervals; no lane-aggregate coverage is claimed.")


# Day 5 B05 — Master-split representative lane risk table

## Conclusion and interpretation

The validated artifact contains one row per **directed core lane and route type** (296 rows across 292 directed lanes). Each row is a representative leg score for that lane-route using the modal observed training departure hour/weekday and median route-specific OSRM time/distance. The B02 lane baseline and history count use all master-training observations on the directed lane that completed by the common cutoff, `2018-09-29 00:00:00`.

B02 residual XGBoost predictions are combined with the saved B03 Method C route-type × calibration-distance-tercile offsets. The reported intervals apply to a representative leg under that constructed context. They do **not** establish coverage for a lane mean, lane aggregate, or operational future distribution. Held-out B02/B03 metrics are reported separately from the lane table.

## Limitations and weather boundary

The representative OSRM values are medians within each lane-route group. `start_hour` and `start_dow` are the modal observed **joint** training context; ties select the smallest hour, then smallest weekday (Monday=0). Lane history is pooled across route types because that is B02's lane-key semantics, while representative OSRM values and departure context are route-specific. Four lanes therefore appear twice, once per observed route type. Distance grouping and reporting both use B02's `osrm_distance_km`; `core_edges.distance_km` is not substituted.

The conformal offsets were calibrated on individual held-out legs. Applying them to a representative feature row does not confer lane-level aggregate coverage. Weather cells below remain exploratory only: they are not inputs to this B02 model, not part of the validated risk table, and their in-sample results are not held-out evaluation.


In [ ]:
# Write-once outputs. All model, conformal, lineage, interval, and in-memory CSV checks passed above.
for output_path in [RISK_PATH, MANIFEST_PATH, SUCCESS_PATH]:
    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output: {output_path}")

DAY5.mkdir(parents=False, exist_ok=True)
_csv_text = risk_table.to_csv(index=False)
with RISK_PATH.open("x", encoding="utf-8", newline="") as f:
    f.write(_csv_text)

# Read back the new artifact and validate its persisted keys and scores before publishing success.
_saved = pd.read_csv(RISK_PATH)
assert len(_saved) == 296 and _saved.duplicated(ROUTE_KEYS).sum() == 0
assert _saved[ROUTE_KEYS].notna().all().all()
assert np.allclose(_saved["pred_ratio"], risk_table["pred_ratio"], rtol=1e-12, atol=1e-12)
assert np.allclose(_saved[["lo50", "hi50", "lo80", "hi80", "lo90", "hi90"]],
                   risk_table[["lo50", "hi50", "lo80", "hi80", "lo90", "hi90"]], rtol=1e-12, atol=1e-12)

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

manifest = {
    "status": "complete",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "artifact": str(RISK_PATH.relative_to(ROOT)).replace("\\", "/"),
    "artifact_sha256": sha256_file(RISK_PATH),
    "lineage": {
        "split": "data/processed/split_trip_ids.csv",
        "training_legs": "data/processed/core_train_legs.csv",
        "all_legs_crosscheck": "data/processed/legs.csv",
        "frozen_directed_lanes": "data/processed/core_edges.csv",
        "prediction_cutoff": "2018-09-29 00:00:00",
        "history_eligibility": "od_end_time <= prediction_timestamp",
        "history_partition": "master train only"
    },
    "model": {
        "file": str(MODEL_PATH.relative_to(ROOT)).replace("\\", "/"),
        "card": str(CARD_PATH.relative_to(ROOT)).replace("\\", "/"),
        "xgboost_version": xgb.__version__,
        "best_iteration": int(model.best_iteration),
        "prediction_api": "XGBRegressor.predict default (best iteration)",
        "features": FEATURES,
        "prediction_parity_rows": int(len(b02_join)),
        "base_pred_max_abs_diff": BASE_MAX_DIFF,
        "residual_max_abs_diff": RESIDUAL_MAX_DIFF,
        "pred_ratio_max_abs_diff": B02_MAX_DIFF,
        "heldout_metrics": B02_METRICS
    },
    "conformal": {
        "parameters": str(PARAMS_PATH.relative_to(ROOT)).replace("\\", "/"),
        "method": conformal_params["method"],
        "distance_field": "osrm_distance_km",
        "distance_edges_km": DISTANCE_EDGES,
        "test_interval_artifact": str(B03_TEST_PATH.relative_to(ROOT)).replace("\\", "/"),
        "test_interval_parity_rows": int(len(b03_join)),
        "heldout_test_coverage": B03_COVERAGE
    },
    "representative_rows": int(len(_saved)),
    "directed_lanes": int(len(EDGE_KEYS)),
    "lane_route_rows": int(len(_saved)),
    "weather_included": False,
    "interpretation": "Representative-leg point scores and Method C intervals; no lane-aggregate coverage claim."
}
with MANIFEST_PATH.open("x", encoding="utf-8", newline="") as f:
    json.dump(manifest, f, indent=2)

success = {
    "status": "complete",
    "manifest": str(MANIFEST_PATH.relative_to(ROOT)).replace("\\", "/"),
    "artifact": str(RISK_PATH.relative_to(ROOT)).replace("\\", "/"),
    "artifact_sha256": manifest["artifact_sha256"],
    "validated_rows": int(len(_saved))
}
with SUCCESS_PATH.open("x", encoding="utf-8", newline="") as f:
    json.dump(success, f, indent=2)
print("DAY 5 B05 STATUS: COMPLETE")
print("Risk table:", RISK_PATH.relative_to(ROOT), "rows:", len(_saved))
print("Manifest:", MANIFEST_PATH.relative_to(ROOT))
print("Success marker:", SUCCESS_PATH.relative_to(ROOT))


In [13]:
import pandas as pd
from pathlib import Path

# 1. Load core train legs and weather daily data
core_train_legs = pd.read_csv("../data/processed/core_train_legs.csv")
weather_daily = pd.read_csv("../data/processed/weather_daily.csv")

print("Core train legs shape:", core_train_legs.shape)
print("Weather daily shape:", weather_daily.shape)

# 2. Preview columns to verify join keys (e.g., hub/center and date)
print("Core train columns:", core_train_legs.columns.tolist()[:6])
print("Weather daily columns:", weather_daily.columns.tolist()[:6])

Core train legs shape: (5395, 15)
Weather daily shape: (2470, 11)
Core train columns: ['trip_uuid', 'source_center', 'destination_center', 'od_start_time', 'od_end_time', 'trip_creation_time']
Weather daily columns: ['date', 'rain_mm', 'precipitation_mm', 'temperature_mean_c', 'hub', 'requested_latitude']


In [14]:
# Convert timestamps to datetime format for merging
core_train_legs["date"] = pd.to_datetime(
    core_train_legs["trip_creation_time"]
).dt.strftime("%Y-%m-%d")
weather_daily["date"] = pd.to_datetime(weather_daily["date"]).dt.strftime(
    "%Y-%m-%d"
)

# Merge weather data onto core train legs using source hub and date
merged_train = pd.merge(
    core_train_legs,
    weather_daily,
    left_on=["source_center", "date"],
    right_on=["hub", "date"],
    how="left",
)

print("Merged core train legs with weather shape:", merged_train.shape)
print("Missing rain_mm values:", merged_train["rain_mm"].isna().sum())
display(merged_train[["trip_uuid", "source_center", "date", "rain_mm"]].head())

Merged core train legs with weather shape: (5395, 26)
Missing rain_mm values: 239


,trip_uuid,source_center,date,rain_mm
0,trip-153671041653548748,IND209304AAA,2018-09-12,1.3
1,trip-153671041653548748,IND462022AAA,2018-09-12,0.3
2,trip-153671043369099517,IND000000ACB,2018-09-12,0.1
3,trip-153671043369099517,IND562132AAA,2018-09-12,2.6
4,trip-153671046011330457,IND400072AAB,2018-09-12,0.6


In [19]:
import numpy as np
from sklearn.metrics import root_mean_squared_error

# Fill missing rain values with 0 (dry days)
merged_train["rain_mm"] = merged_train["rain_mm"].fillna(0)

# Check if target delay ratio column exists (or compute it)
# Let's inspect available columns to ensure we use the correct target name
print(
    "Target / delay columns available:",
    [c for c in merged_train.columns if "ratio" in c or "delay" in c],
)

Target / delay columns available: ['delay_ratio']


In [17]:
print(merged_train.columns.tolist())

['trip_uuid', 'source_center', 'destination_center', 'od_start_time', 'od_end_time', 'trip_creation_time', 'source_name', 'destination_name', 'route_type', 'actual_time_min', 'osrm_time_min', 'osrm_distance_km', 'scan_rows', 'delay_ratio', 'trip_created', 'date', 'rain_mm', 'precipitation_mm', 'temperature_mean_c', 'hub', 'requested_latitude', 'requested_longitude', 'grid_latitude', 'grid_longitude', 'weather_model', 'timezone']


In [18]:
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Fill missing values for the features and target
merged_train["rain_mm"] = merged_train["rain_mm"].fillna(0)
y = merged_train["delay_ratio"].fillna(1.0)

# Define feature sets using the exact column names available
features_base = ["osrm_distance_km", "osrm_time_min"]
features_weather = ["osrm_distance_km", "osrm_time_min", "rain_mm"]

X_base = merged_train[features_base].fillna(0)
X_weather = merged_train[features_weather].fillna(0)

# Train baseline vs weather-augmented models
rf_base = RandomForestRegressor(n_estimators=50, random_state=42)
rf_weather = RandomForestRegressor(n_estimators=50, random_state=42)

rf_base.fit(X_base, y)
rf_weather.fit(X_weather, y)

pred_base = rf_base.predict(X_base)
pred_weather = rf_weather.predict(X_weather)

rmse_base = root_mean_squared_error(y, pred_base)
rmse_weather = root_mean_squared_error(y, pred_weather)

print(f"RMSE without weather: {rmse_base:.4f}")
print(f"RMSE with weather (rain_mm): {rmse_weather:.4f}")
print(
    f"Improvement: {((rmse_base - rmse_weather) / rmse_base) * 100:.2f}% drop in error"
)

RMSE without weather: 0.7014
RMSE with weather (rain_mm): 0.6475
Improvement: 7.69% drop in error


In [ ]:
# EXPLORATORY, IN-SAMPLE (fit and scored on the same rows).
# Not a held-out result. Not used by the delivered lane table.
# Missing rainfall stays NaN; never filled with zero.
# UNRESOLVED: merged_train has per-leg osrm_distance_km and actual_time_min.
# core_edges has lane-level median distance_km and median_actual_time; these
# differ in aggregation level, and actual_time_min is an observed outcome.
# This experiment is not executable with current merged_train columns. Do not
# substitute per-leg fields until the intended lane-level features are confirmed.
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Prepare features (with and without rain)
features_base = ["distance_km", "median_actual_time"]
features_weather = ["distance_km", "median_actual_time", "rain_mm"]

# Exclude rows with a missing target; preserve missing rainfall as NaN.
valid_target = merged_train["delay_ratio"].notna()
X_base = merged_train.loc[valid_target, features_base].fillna(0)
X_weather = merged_train.loc[valid_target, features_weather].copy()
X_weather[features_base] = X_weather[features_base].fillna(0)
y = merged_train.loc[valid_target, "delay_ratio"]

# Quick exploratory in-sample comparison (fit and score on the same rows).
rf_base = RandomForestRegressor(n_estimators=50, random_state=42)
rf_weather = RandomForestRegressor(n_estimators=50, random_state=42)

rf_base.fit(X_base, y)
rf_weather.fit(X_weather, y)

pred_base = rf_base.predict(X_base)
pred_weather = rf_weather.predict(X_weather)

rmse_base = root_mean_squared_error(y, pred_base)
rmse_weather = root_mean_squared_error(y, pred_weather)

print(f"RMSE without weather: {rmse_base:.4f}")
print(f"RMSE with weather (rain_mm): {rmse_weather:.4f}")
print(
    f"Improvement: {((rmse_base - rmse_weather) / rmse_base) * 100:.2f}% drop in"
    " error"
)

# Lane-level deliverables for Person C (appended)

The planning table is `models/outputs/risk_table_lane.csv`; it contains one row for each of the 292 directed core lanes and is generated exclusively from training information. The realized test table is `models/outputs/realized_test_lane.csv` and is evaluation-only. Test outcomes must never influence planning risk values. The evaluation cells verify that the planning table's SHA-256 does not change.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
RISK_SOURCE = ROOT / "experiments/day5/risk_table_master_split_v1.csv"
RISK_MANIFEST = ROOT / "experiments/day5/risk_table_master_split_v1_manifest.json"
TRAIN_PATH = ROOT / "data/processed/core_train_legs.csv"
EDGE_PATH = ROOT / "data/processed/core_edges.csv"
SPLIT_PATH = ROOT / "data/processed/split_trip_ids.csv"
OUT_DIR = ROOT / "models/outputs"
RISK_OUT = OUT_DIR / "risk_table_lane.csv"
EXPECTED_SOURCE_SHA = "9d0d697addfe53ad6b1d3267161c0a6ba9728bcfa1ccb7cee8449e01f098a498"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert not RISK_OUT.exists(), f"Refusing to overwrite existing output: {RISK_OUT}"
for p in (RISK_SOURCE, RISK_MANIFEST, TRAIN_PATH, EDGE_PATH, SPLIT_PATH):
    assert p.is_file(), f"Required input missing: {p}"
source_sha = sha256_file(RISK_SOURCE)
manifest = json.loads(RISK_MANIFEST.read_text(encoding="utf-8"))
assert source_sha == EXPECTED_SOURCE_SHA, (source_sha, EXPECTED_SOURCE_SHA)
assert manifest["artifact_sha256"].lower() == EXPECTED_SOURCE_SHA

representatives = pd.read_csv(RISK_SOURCE)
required_rep = [
    "source_center", "destination_center", "route_type", "lane_base",
    "lane_history_count", "representative_training_legs", "pred_ratio",
    "lo50", "hi50", "lo90", "hi90"
]
missing_rep = sorted(set(required_rep) - set(representatives.columns))
assert not missing_rep, f"Representative table missing fields: {missing_rep}"
assert len(representatives) == 296
rep_lanes = set(map(tuple, representatives[["source_center", "destination_center"]].drop_duplicates().to_numpy()))

train = pd.read_csv(TRAIN_PATH)
edges = pd.read_csv(EDGE_PATH)
split = pd.read_csv(SPLIT_PATH, usecols=["trip_uuid", "split"])
required_train = ["trip_uuid", "source_center", "destination_center", "route_type", "osrm_distance_km", "delay_ratio"]
missing_train = sorted(set(required_train) - set(train.columns))
assert not missing_train, f"Training legs missing fields: {missing_train}"
edge_keys = ["source_center", "destination_center"]
assert set(edge_keys).issubset(edges.columns)
edge_pairs = edges[edge_keys].drop_duplicates()
assert len(edge_pairs) == 292, f"Expected 292 unique core lanes, got {len(edge_pairs)}"
assert not edges.duplicated(edge_keys).any(), "Unexpected duplicate lane identifiers in core edge set"
core_lanes = set(map(tuple, edge_pairs.to_numpy()))
assert core_lanes == rep_lanes, f"Representative/core lane mismatch: missing={len(core_lanes-rep_lanes)}, extra={len(rep_lanes-core_lanes)}"
assert split["trip_uuid"].is_unique
split_map = split.set_index("trip_uuid")["split"]
train_labels = train["trip_uuid"].map(split_map)
assert train_labels.notna().all(), f"Training UUIDs missing from split: {int(train_labels.isna().sum())}"
assert train_labels.eq("train").all(), f"Non-train rows in core_train_legs: {train.loc[~train_labels.eq('train'), 'trip_uuid'].head().tolist()}"
assert train["delay_ratio"].notna().all(), "Training delay_ratio contains missing values"

lane_stats = train.groupby(edge_keys, sort=False).agg(
    n_train_legs=("delay_ratio", "size"),
    lane_median=("delay_ratio", "median")
).reset_index()
lane_n = lane_stats.set_index(edge_keys)["n_train_legs"]
lane_medians = lane_stats.set_index(edge_keys)["lane_median"]
assert set(map(tuple, lane_stats[edge_keys].to_numpy())).issubset(core_lanes)

# Precompute fallback medians on training legs only, using the lane's median OSRM distance.
def distance_bucket(x):
    if x <= 31.21256666666666:
        return "short"
    if x <= 60.86566666666664:
        return "medium"
    return "long"

train["__bucket"] = train["osrm_distance_km"].map(distance_bucket)
global_median = float(train["delay_ratio"].median())
group_medians = train.groupby(["route_type", "__bucket"])["delay_ratio"].median()
lane_distance = train.groupby(edge_keys)["osrm_distance_km"].median()
lane_route = train.groupby(edge_keys)["route_type"].agg(
    lambda s: sorted(s.dropna().value_counts().items(), key=lambda kv: (-kv[1], str(kv[0])))[0][0]
)

rep_by_lane = representatives.groupby(edge_keys, sort=False)
base_records = []
for lane in sorted(core_lanes):
    n = int(lane_n.get(lane, 0))
    lane_med = float(lane_medians.get(lane, np.nan))
    if n >= 3:
        base, base_source = lane_med, "lane"
        lane_rows = representatives[
            representatives["source_center"].eq(lane[0]) &
            representatives["destination_center"].eq(lane[1])
        ]
        assert np.allclose(lane_rows["lane_base"].to_numpy(dtype=float), lane_med, rtol=0, atol=1e-12), f"lane_base mismatch for {lane}"
        assert lane_rows["lane_history_count"].eq(n).all(), f"lane_history_count mismatch for {lane}"
        assert float(lane_rows["representative_training_legs"].sum()) == n, f"representative weights do not sum to n_train_legs for {lane}"
    else:
        route = lane_route.get(lane, None)
        dist = float(lane_distance.get(lane, np.nan))
        assert route is not None and np.isfinite(dist), f"Cannot determine fallback route/distance for {lane}"
        bucket = distance_bucket(dist)
        key = (route, bucket)
        if key in group_medians.index and pd.notna(group_medians.loc[key]):
            base, base_source = float(group_medians.loc[key]), "route_distance_group"
        else:
            base, base_source = global_median, "global"
        if base_source == "route_distance_group":
            assert np.isclose(base, float(group_medians.loc[key]), rtol=0, atol=1e-12)
        else:
            assert np.isclose(base, global_median, rtol=0, atol=1e-12)
    assert np.isfinite(base) and base > 0, f"Invalid base for {lane}: {base}"
    base_records.append({"source_center": lane[0], "destination_center": lane[1],
                        "lane_base": base, "base_source": base_source, "n_train_legs": n})

bases = pd.DataFrame(base_records)
prediction_fields = ["pred_ratio", "lo50", "hi50", "lo90", "hi90"]
weights = representatives["representative_training_legs"].to_numpy(dtype=float)
assert np.isfinite(weights).all() and (weights > 0).all(), "Representative weights must be finite and positive"
for col in prediction_fields:
    assert np.isfinite(representatives[col].to_numpy(dtype=float)).all(), f"Non-finite source values in {col}"

agg_records = []
for lane, rows in representatives.groupby(edge_keys, sort=False):
    w = rows["representative_training_legs"].to_numpy(dtype=float)
    assert np.isfinite(w).all() and (w > 0).all() and w.sum() > 0, f"Invalid weights for {lane}"
    vals = {col: float(np.average(rows[col].to_numpy(dtype=float), weights=w)) for col in prediction_fields}
    agg_records.append({"source_center": lane[0], "destination_center": lane[1], **vals})
aggregated = pd.DataFrame(agg_records)
risk = bases.merge(aggregated, on=edge_keys, how="left", validate="one_to_one")
assert risk[prediction_fields].notna().all().all(), "Missing aggregated prediction values"
for mult, src in [("mult_pred", "pred_ratio"), ("mult_lo90", "lo90"), ("mult_hi90", "hi90"), ("mult_lo50", "lo50"), ("mult_hi50", "hi50")]:
    risk[mult] = risk[src] / risk["lane_base"]

risk = risk[[
    "source_center", "destination_center", "lane_base", "base_source", "n_train_legs",
    "mult_pred", "mult_lo90", "mult_hi90", "mult_lo50", "mult_hi50"
]].sort_values(edge_keys, kind="mergesort").reset_index(drop=True)

assert len(risk) == 292
assert risk[edge_keys].drop_duplicates().shape[0] == 292
assert set(map(tuple, risk[edge_keys].to_numpy())) == core_lanes
assert not risk.isna().any().any()
numeric_cols = [c for c in risk.columns if c not in edge_keys + ["base_source"]]
assert np.isfinite(risk[numeric_cols].to_numpy(dtype=float)).all()
assert (risk[["lane_base", "mult_pred", "mult_lo90", "mult_hi90", "mult_lo50", "mult_hi50"]] > 0).all().all()
valid90 = (risk["mult_lo90"] <= risk["mult_pred"]) & (risk["mult_pred"] <= risk["mult_hi90"])
valid50 = (risk["mult_lo90"] <= risk["mult_lo50"]) & (risk["mult_lo50"] <= risk["mult_pred"]) & (risk["mult_pred"] <= risk["mult_hi50"]) & (risk["mult_hi50"] <= risk["mult_hi90"])
assert valid90.all(), f"90% interval order failures:\n{risk.loc[~valid90].to_string(index=False)}"
assert valid50.all(), f"Nested interval order failures:\n{risk.loc[~valid50].to_string(index=False)}"
width_std = float((risk["mult_hi90"] - risk["mult_lo90"]).std())
assert width_std > 0
assert not ({"trip_uuid", "y_ratio", "delay_ratio", "actual_time_min"} & set(risk.columns))

OUT_DIR.mkdir(parents=True, exist_ok=True)
risk.to_csv(RISK_OUT, index=False, mode="x")
risk_sha = sha256_file(RISK_OUT)
print("source_sha256:", source_sha)
print("training_rows:", len(train))
print("official_train_label_rows:", int(train_labels.eq("train").sum()))
print("core_lanes:", len(core_lanes))
print("risk_rows:", len(risk))
print("columns:", risk.columns.tolist())
print("base_source_counts:", risk["base_source"].value_counts().sort_index().to_dict())
print("lane_width_std:", width_std)
print("risk_table_sha256:", risk_sha)
print("risk_table_first_five_rows:")
print(risk.head(5).to_string(index=False))
print("representative_source_lanes_with_multiple_route_rows:")
dup_lanes = representatives.groupby(edge_keys).size()
for lane in sorted(dup_lanes[dup_lanes > 1].index.tolist()):
    print(representatives[
        representatives["source_center"].eq(lane[0]) &
        representatives["destination_center"].eq(lane[1])
    ].to_string(index=False))


source_sha256: 9d0d697addfe53ad6b1d3267161c0a6ba9728bcfa1ccb7cee8449e01f098a498
training_rows: 5395
official_train_label_rows: 5395
core_lanes: 292
risk_rows: 292
columns: ['source_center', 'destination_center', 'lane_base', 'base_source', 'n_train_legs', 'mult_pred', 'mult_lo90', 'mult_hi90', 'mult_lo50', 'mult_hi50']
base_source_counts: {'lane': 292}
lane_width_std: 0.7367854519992618
risk_table_sha256: d32af6b38157cb9ecd4a99214b9fba39c0fdc4b2b7a7a87d98e0302439990d7e
risk_table_first_five_rows:
source_center destination_center  lane_base base_source  n_train_legs  mult_pred  mult_lo90  mult_hi90  mult_lo50  mult_hi50
 IND000000AAL       IND411033AAA   2.570833        lane            14   1.013161   0.615944   2.603782   0.903510   1.255802
 IND000000ACA       IND144001AAB   1.612245        lane            13   1.024786   0.746434   1.719414   0.940344   1.135453
 IND000000ACA       IND160002AAC   2.064516        lane            59   1.013022   0.793621   1.555958   0.946199   1.09989

In [2]:
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
LEGS_PATH = ROOT / "data/processed/legs.csv"
SPLIT_PATH = ROOT / "data/processed/split_trip_ids.csv"
EDGE_PATH = ROOT / "data/processed/core_edges.csv"
RISK_OUT = ROOT / "models/outputs/risk_table_lane.csv"
REALIZED_OUT = ROOT / "models/outputs/realized_test_lane.csv"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert RISK_OUT.is_file()
assert not REALIZED_OUT.exists(), f"Refusing to overwrite existing output: {REALIZED_OUT}"
risk_sha_before = sha256_file(RISK_OUT)
split = pd.read_csv(SPLIT_PATH, usecols=["trip_uuid", "split"])
test_ids = set(split.loc[split["split"].eq("test"), "trip_uuid"])
legs = pd.read_csv(LEGS_PATH)
assert {"trip_uuid", "source_center", "destination_center", "delay_ratio"}.issubset(legs.columns)
test_legs = legs.loc[legs["trip_uuid"].isin(test_ids)].copy()
assert test_legs["delay_ratio"].notna().all(), "Missing delay_ratio on test rows"
edges = pd.read_csv(EDGE_PATH, usecols=["source_center", "destination_center"])
core_keys = set(map(tuple, edges.drop_duplicates().to_numpy()))
test_legs["__lane"] = list(zip(test_legs["source_center"], test_legs["destination_center"]))
core_test = test_legs.loc[test_legs["__lane"].isin(core_keys)].copy()
lane_counts = core_test.groupby(["source_center", "destination_center"]).size().rename("n_test_legs")
lane_means = core_test.groupby(["source_center", "destination_center"])["delay_ratio"].mean().rename("realized_ratio_mean")
observed = pd.concat([lane_counts, lane_means], axis=1).reset_index()
risk = pd.read_csv(RISK_OUT, usecols=["source_center", "destination_center", "lane_base"])
realized = observed.merge(risk, on=["source_center", "destination_center"], how="left", validate="one_to_one")
assert realized["lane_base"].notna().all()
included = realized.loc[realized["n_test_legs"].ge(5)].copy()
included["realized_mult"] = included["realized_ratio_mean"] / included["lane_base"]
included = included[["source_center", "destination_center", "n_test_legs", "realized_ratio_mean", "realized_mult"]]
assert not included.isna().any().any()
assert included["n_test_legs"].ge(5).all()
assert set(map(tuple, included[["source_center", "destination_center"]].to_numpy())).issubset(core_keys)
assert np.isfinite(included.select_dtypes(include=[np.number]).to_numpy(dtype=float)).all()
included.to_csv(REALIZED_OUT, index=False, mode="x")
risk_sha_after = sha256_file(RISK_OUT)
assert risk_sha_before == risk_sha_after, f"Risk table changed during evaluation: before={risk_sha_before}, after={risk_sha_after}"
realized_sha = sha256_file(REALIZED_OUT)
n_test_legs = len(core_test)
n_any = int((lane_counts > 0).sum())
n_ge5 = int((lane_counts >= 5).sum())
print("total_core_test_legs:", n_test_legs)
print("lanes_with_at_least_one_test_leg:", n_any)
print("lanes_with_at_least_five_test_legs:", n_ge5)
print("lanes_omitted_below_five_or_zero:", 292 - n_ge5)
print("rows_written:", len(included))
print("columns:", included.columns.tolist())
print("realized_test_lane_sha256:", realized_sha)
print("risk_table_sha256_before:", risk_sha_before)
print("risk_table_sha256_after:", risk_sha_after)
print("risk_table_sha_unchanged:", risk_sha_before == risk_sha_after)
print("realized_test_lane_first_ten_rows:")
print(included.head(10).to_string(index=False))


total_core_test_legs: 1185
lanes_with_at_least_one_test_leg: 226
lanes_with_at_least_five_test_legs: 93
lanes_omitted_below_five_or_zero: 199
rows_written: 93
columns: ['source_center', 'destination_center', 'n_test_legs', 'realized_ratio_mean', 'realized_mult']
realized_test_lane_sha256: 8a844d81dbe5bd58513ff6ae4d8d931bb6848c15ab1cfa1cf777775a50b36b84
risk_table_sha256_before: d32af6b38157cb9ecd4a99214b9fba39c0fdc4b2b7a7a87d98e0302439990d7e
risk_table_sha256_after: d32af6b38157cb9ecd4a99214b9fba39c0fdc4b2b7a7a87d98e0302439990d7e
risk_table_sha_unchanged: True
realized_test_lane_first_ten_rows:
source_center destination_center  n_test_legs  realized_ratio_mean  realized_mult
 IND000000ACA       IND160002AAC            7             2.087671       1.011216
 IND000000ACB       IND110037AAK            7             5.253821       2.927129
 IND000000ACB       IND110064AAA            6             1.620603       1.008743
 IND000000ACB       IND110095AAB            5             2.385053    

In [3]:
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = Path.cwd()
PRED_PATH = ROOT / "data/processed/preds_cal_test_master_split_v1.csv"
INTERVAL_PATH = ROOT / "experiments/day3/master_split_v2/final/interval_comparison_master_split_v2.csv"
BOOT_PATH = ROOT / "experiments/day3/master_split_v2/evaluation/method_c_trip_bootstrap_coverage_ci.csv"
RISK_OUT = ROOT / "models/outputs/risk_table_lane.csv"
EXPECTED_PRED_SHA = "5165F4027AFEA3491F5BE01D6D5D22A1B4318FF486270EB9EFA70BC1A8777EB4"
EXPECTED_MODEL_MAE = 0.653957215972931
EXPECTED_COV90 = 0.8884842967683204

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

risk_sha_before = sha256_file(RISK_OUT)
pred_sha = sha256_file(PRED_PATH)
assert pred_sha.lower() == EXPECTED_PRED_SHA.lower(), (pred_sha, EXPECTED_PRED_SHA)
preds = pd.read_csv(PRED_PATH)
required = {"pile", "pred_ratio", "base_pred", "y_ratio"}
assert required.issubset(preds.columns), f"Prediction columns missing: {sorted(required-set(preds.columns))}"
test = preds.loc[preds["pile"].eq("test")].copy()
assert len(test) == 4394, f"Expected 4,394 test rows, got {len(test)}"
assert test[list(required - {"pile"})].notna().all().all()
model_mae = mean_absolute_error(test["y_ratio"], test["pred_ratio"])
model_rmse = mean_squared_error(test["y_ratio"], test["pred_ratio"]) ** 0.5
base_mae = mean_absolute_error(test["y_ratio"], test["base_pred"])
base_rmse = mean_squared_error(test["y_ratio"], test["base_pred"]) ** 0.5
assert np.isclose(model_mae, EXPECTED_MODEL_MAE, rtol=0, atol=1e-9), f"Model MAE mismatch: {model_mae}"

intervals = pd.read_csv(INTERVAL_PATH)
chosen = intervals.loc[intervals["method"].eq("C grouped (route x distance)")].copy()
cov = chosen.set_index("level")["coverage"].to_dict()
coverage_50, coverage_80, coverage_90 = float(cov["50%"]), float(cov["80%"]), float(cov["90%"])
assert np.isclose(coverage_90, EXPECTED_COV90, rtol=0, atol=1e-9), f"90% coverage mismatch: {coverage_90}"
bootstrap = pd.read_csv(BOOT_PATH)
boot = bootstrap.loc[bootstrap["level"].isin(["50%", "80%", "90%"]), ["level", "coverage", "trip_bootstrap_95_lo", "trip_bootstrap_95_hi"]].sort_values("level")

print("prediction_sha256:", pred_sha)
print("test_rows:", len(test))
print("model MAE:", model_mae)
print("model RMSE:", model_rmse)
print("base_pred MAE:", base_mae)
print("base_pred RMSE:", base_rmse)
print("Method C coverage label: leg-level held-out test coverage; not lane-day coverage; not nominal coverage")
print("50% coverage:", coverage_50)
print("80% coverage:", coverage_80)
print("90% coverage:", coverage_90)
print("Trip-bootstrap 95% confidence bounds:")
print(boot.to_string(index=False))
print("risk_table_sha256_before:", risk_sha_before)
print("risk_table_sha256_after:", sha256_file(RISK_OUT))
assert risk_sha_before == sha256_file(RISK_OUT)


prediction_sha256: 5165f4027afea3491f5be01d6d5d22a1b4318ff486270eb9efa70bc1a8777eb4
test_rows: 4394
model MAE: 0.653957215972931
model RMSE: 1.727802656778531
base_pred MAE: 0.6686060922605898
base_pred RMSE: 1.737589550840338
Method C coverage label: leg-level held-out test coverage; not lane-day coverage; not nominal coverage
50% coverage: 0.4615384615384615
80% coverage: 0.7762858443331816
90% coverage: 0.8884842967683204
Trip-bootstrap 95% confidence bounds:
level  coverage  trip_bootstrap_95_lo  trip_bootstrap_95_hi
  50%  0.461538                 0.446                 0.477
  80%  0.776286                 0.764                 0.789
  90%  0.888484                 0.879                 0.898
risk_table_sha256_before: d32af6b38157cb9ecd4a99214b9fba39c0fdc4b2b7a7a87d98e0302439990d7e
risk_table_sha256_after: d32af6b38157cb9ecd4a99214b9fba39c0fdc4b2b7a7a87d98e0302439990d7e


In [4]:
import pandas as pd

r = pd.read_csv("models/outputs/risk_table_lane.csv")

print(
    len(r),
    r[["source_center", "destination_center"]]
    .drop_duplicates().shape[0],
    r.isna().sum().sum()
)

print(
    (
        (r.mult_lo90 > 0)
        & (r.mult_lo90 <= r.mult_pred)
        & (r.mult_pred <= r.mult_hi90)
    ).all()
)

print((r.mult_hi90 - r.mult_lo90).describe())


292 292 0

True

count    292.000000
mean       1.380638
std        0.736785
min        0.398432
25%        0.818133
50%        1.028122
75%        1.826028
max        4.303495
dtype: float64


In [5]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

risk_check = pd.read_csv("models/outputs/risk_table_lane.csv")
realized_check = pd.read_csv("models/outputs/realized_test_lane.csv")
raw_width = risk_check["mult_hi90"] - risk_check["mult_lo90"]
distinct_widths = int(raw_width.nunique())
width_min, width_median, width_max = map(float, [raw_width.min(), raw_width.median(), raw_width.max()])
one_test_lanes = int((lane_counts > 0).sum())
five_test_lanes = int((lane_counts >= 5).sum())
base_counts = risk_check["base_source"].value_counts().to_dict()
all_col_meanings = {
    "source_center / destination_center": "directed lane endpoints",
    "lane_base": "training-only lane baseline delay ratio, with documented sparse-lane fallbacks",
    "base_source": "which training-only baseline rule supplied lane_base",
    "n_train_legs": "number of core training legs on the lane across route types",
    "mult_pred": "weighted representative-route predicted ratio divided by lane_base",
    "mult_lo90 / mult_hi90": "weighted representative-route 90% interval endpoints divided by lane_base",
    "mult_lo50 / mult_hi50": "weighted representative-route 50% interval endpoints divided by lane_base",
}
computed_note = f"""### Computed interval method and limitations

**Planning table columns.** {all_col_meanings}

All `mult_*` values are dimensionless ratios calculated as a ratio divided by `lane_base`. `lane_base` is the lane training median when at least three training legs are available; otherwise it uses the route-type × distance-bucket training median; otherwise it uses the global training median. `base_source` is one of `lane`, `route_distance_group`, or `global`. The observed source counts are {base_counts}.

Representative route rows are combined by a weighted arithmetic mean, using `representative_training_legs` as the weight for each point prediction and interval endpoint. Averaging interval endpoints is **not a formally valid mixture interval**.

Method C held-out results are leg-level held-out test coverage: not lane-day coverage; not nominal coverage. Coverage was {coverage_50:.6f} at 50%, {coverage_80:.6f} at 80%, and {coverage_90:.6f} at 90%. The observed 90% interval coverage is **88.85%, which is not 90% and is not lane-day coverage**.

The scaled 90% width has {distinct_widths} distinct raw values; minimum, median and maximum are {width_min:.6f}, {width_median:.6f}, and {width_max:.6f}. Width variation is inherited from route-type × distance-group intervals, route-row aggregation and lane-base scaling. It is a proxy, not lane-specific calibrated uncertainty.

Predictions represent one leg per lane-route at cutoff `2018-09-29 00:00`, from B02 residual XGBoost without weather. Weather is an optional separate experiment and is not used in these deliverables.

`realized_test_lane.csv` is evaluation-only and must never be used as a planning input. {one_test_lanes} lanes had at least one test leg; {five_test_lanes} lanes with at least five test legs were included. `pred_excess_h` is not delivered. These conclusions are limited by approximately three weeks of 2018 data and a short test window. No formal lane-day coverage or calibrated lane-closure probabilities are claimed.
"""
display(Markdown(computed_note))


### Computed interval method and limitations

**Planning table columns.** {'source_center / destination_center': 'directed lane endpoints', 'lane_base': 'training-only lane baseline delay ratio, with documented sparse-lane fallbacks', 'base_source': 'which training-only baseline rule supplied lane_base', 'n_train_legs': 'number of core training legs on the lane across route types', 'mult_pred': 'weighted representative-route predicted ratio divided by lane_base', 'mult_lo90 / mult_hi90': 'weighted representative-route 90% interval endpoints divided by lane_base', 'mult_lo50 / mult_hi50': 'weighted representative-route 50% interval endpoints divided by lane_base'}

All `mult_*` values are dimensionless ratios calculated as a ratio divided by `lane_base`. `lane_base` is the lane training median when at least three training legs are available; otherwise it uses the route-type × distance-bucket training median; otherwise it uses the global training median. `base_source` is one of `lane`, `route_distance_group`, or `global`. The observed source counts are {'lane': 292}.

Representative route rows are combined by a weighted arithmetic mean, using `representative_training_legs` as the weight for each point prediction and interval endpoint. Averaging interval endpoints is **not a formally valid mixture interval**.

Method C held-out results are leg-level held-out test coverage: not lane-day coverage; not nominal coverage. Coverage was 0.461538 at 50%, 0.776286 at 80%, and 0.888484 at 90%. The observed 90% interval coverage is **88.85%, which is not 90% and is not lane-day coverage**.

The scaled 90% width has 289 distinct raw values; minimum, median and maximum are 0.398432, 1.028122, and 4.303495. Width variation is inherited from route-type × distance-group intervals, route-row aggregation and lane-base scaling. It is a proxy, not lane-specific calibrated uncertainty.

Predictions represent one leg per lane-route at cutoff `2018-09-29 00:00`, from B02 residual XGBoost without weather. Weather is an optional separate experiment and is not used in these deliverables.

`realized_test_lane.csv` is evaluation-only and must never be used as a planning input. 226 lanes had at least one test leg; 93 lanes with at least five test legs were included. `pred_excess_h` is not delivered. These conclusions are limited by approximately three weeks of 2018 data and a short test window. No formal lane-day coverage or calibrated lane-closure probabilities are claimed.